# Audio Library Python ETL Project
The purpose of this python project is to create a managed audio library, where audio files will be uploaded and stored.  The library will also enable the user to easily manage metadata by adding, deleting, or copying metadata.  The library will also enable the user to easily search for specific audio files by filtering keywords embedded in metadata.

Parameters of the project:
- The project audio files will be in the following format:
  - .aiff, which is the industry standard for professional asset delivery to music libraries for high quality uncompressed audio, detailed and consistent storage for metadata, portability.
  - 24/48k
  - 2 channel
  - 5 second length
- In keeping with workflow similar to DISCO metadata management
  - Audio files will be created offline on the client side.
  - Uploaded to server with bulk upload process.
  - Metadata will be managed (added, copied, deleted) by the user.  This includes song and artist name, album, album art, keyword descriptors, etc.
-Display of metadata
  - tiered display, depending on view
  - View of all albums, artists, etc. will display limited information relevant to a cursory view:
    - Artist
    - Album
    - Album art
  - Viewing a specific album may list the same, but also:
    - track names
    - length
- Query and filter by name, keywords, lyrics, etc.

Goals include:
- Create a client
- Create a bucket or connect to existing S3 bucket
- Query for contents of bucket (the bucket itself, name, contents by folder), and clean up the output
- Create/upload .aif files to S3 bucket by folder.
- Create, copy, edit, delete metadata on audio files.
- Display metadata in a clean, useful way, including album art Attached PICture (APIC)


# Catalog Prep
The following code is commented out, but would be otherwise used to generate a folder with a random name that represents a band and album name.  The folder is filled with 10 random named audio files in the following format:
- .aiff
- 24/48k
- 2 channel
- 5 second length

In [1]:
'''
#Create empty .aif files using torchaudio
import os
import torchaudio
import torch
import randomname
#Parameters
sample_rate = 44100
duration_seconds = 5
num_channels = 2
bits_per_sample = 24  # Specify the bit depth (e.g., 16, 24, 32)
def create_empty_aif(file_path, sample_rate, num_channels, bits_per_sample):
    # Create an empty waveform with zeros
    duration_samples = int(sample_rate * duration_seconds)
    waveform = torch.zeros(num_channels, duration_samples)
    # Save the empty waveform as an .aif file
    torchaudio.save(file_path, waveform, sample_rate, bits_per_sample=bits_per_sample)

for i in range(9): #change to number of folders you want to create

    # Generate folder name as "Band - Album"
    band_name = randomname.get_name().replace('-', ' ').title()
    album_name = randomname.get_name().replace('-', ' ').title()
    subfolder_name = f"{band_name} - {album_name}"

    #Set ouptut directory inside the existing "Catalog" folder
    output_directory = os.path.join("Catalog", subfolder_name)

    os.makedirs(output_directory, exist_ok=True)

    for i in range(10):
        file_name = randomname.get_name().replace('-', ' ').title()
        file_path = os.path.join(output_directory, f"{file_name}.aif")
        create_empty_aif(file_path, sample_rate, num_channels, bits_per_sample)

    print(f"Created 10 empty .aif files in directory: {output_directory}")
'''

'\n#Create empty .aif files using torchaudio\nimport os\nimport torchaudio\nimport torch\nimport randomname\n#Parameters\nsample_rate = 44100\nduration_seconds = 5\nnum_channels = 2\nbits_per_sample = 24  # Specify the bit depth (e.g., 16, 24, 32)\ndef create_empty_aif(file_path, sample_rate, num_channels, bits_per_sample):\n    # Create an empty waveform with zeros\n    duration_samples = int(sample_rate * duration_seconds)\n    waveform = torch.zeros(num_channels, duration_samples)\n    # Save the empty waveform as an .aif file\n    torchaudio.save(file_path, waveform, sample_rate, bits_per_sample=bits_per_sample)\n\nfor i in range(9): #change to number of folders you want to create\n\n    # Generate folder name as "Band - Album"\n    band_name = randomname.get_name().replace(\'-\', \' \').title()\n    album_name = randomname.get_name().replace(\'-\', \' \').title()\n    subfolder_name = f"{band_name} - {album_name}"\n\n    #Set ouptut directory inside the existing "Catalog" folder\n

# Create a client

In [2]:
import boto3
from botocore.exceptions import ClientError

In [3]:
# Create a client
client = boto3.client('s3', region_name='us-west-2')

# Create a bucket
Here a bucket will be created.  We will first inquire if the bucket exists.  If it does not exist, a bucket will be created.  Otherwise, a message will print indicating the bucket already exists.

In [4]:


#Inquire if a specific bucket exists, and create it if it does not exist. If the bucket exists, print a message to the user.


def bucket_exists(bucket_name):
    """Check if an S3 bucket exists

    :param bucket_name: Name of the bucket to check
    :return: True if bucket exists, else False
    """
    s3_client = boto3.client('s3')
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        return True
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == '404':
            return False
        else:
            raise e


def create_bucket(bucket_name, region=None):
    """Create an S3 bucket in a specified region

    If a region is not specified, the bucket is created in the S3 default
    region (us-east-1).

    :param bucket_name: Bucket to create
    :param region: String region to create bucket in, e.g., 'us-west-2'
    :return: True if bucket created, else False
    """

    # Create bucket
    try:
        s3_client = boto3.client('s3')

        create_params = {'Bucket': bucket_name}

        if region and region != 'us-west-2':
            create_params['CreateBucketConfiguration'] = {
                'LocationConstraint': region
            }

        #Create the bucket
        s3_client.create_bucket(**create_params)
        print(f"Bucket '{bucket_name}' created successfully.")
        return True
    except ClientError as e:
        print(f"Error creating bucket: {e}")
        return False

bucket_name = 'audio-catalog-bucket-2026'  # Change to your desired bucket name
region = 'us-east-2'


if bucket_exists(bucket_name):#check if bucket exists before creating it
    print(f"Bucket '{bucket_name}' already exists.")#print message if bucket exists
else:#otherwise, create the bucket
    create_bucket(bucket_name, region)


Bucket 'audio-catalog-bucket-2026' already exists.


# Upload Files and folders
Upload files into their respective folders

In [5]:
import logging
import boto3
from botocore.exceptions import ClientError
import os

def upload_file(file_name, bucket_name, object_name=None):
    """Upload a file to an S3 bucket

    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param object_name: S3 object name. If not specified then file_name is used
    :return: True if file was uploaded, else False
    """

    # If S3 object_name was not specified, use file_name
    if object_name is None:
        object_name = os.path.basename(file_name)

    # Upload the file
    s3_client = boto3.client('s3')
    try:
        response = s3_client.upload_file(file_name, bucket_name, object_name)
    except ClientError as e:
        logging.error(e)
        return False
    return True

s3 = boto3.client('s3', region_name='us-west-2')
folder = 'Catalog'
folders = [item for item in os.listdir(folder) if os.path.isdir(os.path.join(folder, item))]
print(folders)

for folder in folders:
    files = [item for item in os.listdir(os.path.join('Catalog', folder)) if os.path.isfile(os.path.join('Catalog', folder, item))]
    print(files)    
    for file in files:
        local_path = os.path.join('Catalog', folder, file)
        s3_key = os.path.join(folder, file).replace('\\', '/')
        upload_file(local_path, bucket_name, s3_key)

['Ambitious Archive - Forward Sweeper', 'Cold Heater - Stable Flower', 'Composite Proton - Slow Buyer', 'Finite Color - Electrical Wingman', 'Genteel Avenue - Counting Jackal', 'Iron Basket - Steamed Baseline', 'Salty Name - Molten Ocelot', 'Sunny Cornice - Spatial Flamingo', 'The Empty Heads - The Void and Beyond', 'Woolen Dividend - Magenta Granite']
['Ash Director.aif', 'Bounded Entropy.aif', 'Forward Dope.aif', 'Icy Riser.aif', 'Icy Skier.aif', 'Internal Design.aif', 'Metallic Technology.aif', 'Oblique Mead.aif', 'Relaxed Hardcover.aif', 'Rubbery Hamburger.aif']
['Acidic Form.aif', 'Bitter Sidewalk.aif', 'Happy Time.aif', 'Potential Innovation.aif', 'Salty Canon.aif', 'Sensitive Purse.aif', 'Smoked Tower.aif', 'Sour Pagoda.aif', 'Stern Rapids.aif', 'Tangy Mamba.aif']
['Bland Lane.aif', 'Dry Ghoul.aif', 'Fermented Scene.aif', 'Gentle Ravvet.aif', 'Muffled Style.aif', 'Shabby Mono.aif', 'Sunny Queue.aif', 'Tough Sitrep.aif', 'White Mat.aif', 'Wise Particle.aif']
['Advanced Bell.aif',

# Listing Buckets and contents
Create a client session, paginator, and page iterator using two methods.  
- The first method uses list_objects_v2, but the bucket is specified.
- The second method uses the list_buckets paginator method, and just lists buckets, and is false if nothing found.

In [6]:


# Create a reusable Paginator
#list_objects_v2 method
paginator = client.get_paginator('list_objects_v2')

# Create a PageIterator from the Paginator
page_iterator = paginator.paginate(Bucket='justin-music-ai-experiments-2026')

for page in page_iterator:
    print(page['Name'])
    print(page['Contents']) #List the contents in page['Contents']
    

#list_buckets paginator method

paginator = client.get_paginator('list_buckets')
response_iterator = paginator.paginate(
    PaginationConfig={
        "PageSize": 50,
        "StartingToken": None,
    }
)
buckets_found = False
for page in response_iterator:
    if "Buckets" in page and page["Buckets"]:  #Check if "Buckets" is in page dictionary.  If so, check if non-empty.
        buckets_found = True
        for bucket in page["Buckets"]:
            print(f"\t{bucket['Name']}")
if not buckets_found:
    print("No buckets found")


justin-music-ai-experiments-2026
[{'Key': 'The Empty Heads - The Void and Beyond/Drained.aif', 'LastModified': datetime.datetime(2026, 2, 10, 5, 35, 44, tzinfo=tzutc()), 'ETag': '"e3774f789d3b239ccbfeec03b27a0a91"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 882054, 'StorageClass': 'STANDARD'}, {'Key': 'The Empty Heads - The Void and Beyond/Glass Half Full.aif', 'LastModified': datetime.datetime(2026, 2, 10, 5, 35, 44, tzinfo=tzutc()), 'ETag': '"e3774f789d3b239ccbfeec03b27a0a91"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 882054, 'StorageClass': 'STANDARD'}, {'Key': 'The Empty Heads - The Void and Beyond/Hardly There.aif', 'LastModified': datetime.datetime(2026, 2, 10, 5, 35, 44, tzinfo=tzutc()), 'ETag': '"e3774f789d3b239ccbfeec03b27a0a91"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 882054, 'StorageClass': 'STANDARD'}, {'Key': 'The Empty Heads - The Void and Beyond/Hole in the Bucket.aif',

# A simpler method using list_buckets()

In [7]:
bucket_list = client.list_buckets()
print(bucket_list)

{'ResponseMetadata': {'RequestId': '4T00ZMXSBKYAPNYK', 'HostId': '8B47+NqxpwqjdUYCnyZtRp2aqNCvGPe1FLmVX1ax5bFTHvJtArGoxW8+zx9luRfCDlHjyr6XTf0D+KvpPrCKECNZrg8h6LANdNeeWOoouaI=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '8B47+NqxpwqjdUYCnyZtRp2aqNCvGPe1FLmVX1ax5bFTHvJtArGoxW8+zx9luRfCDlHjyr6XTf0D+KvpPrCKECNZrg8h6LANdNeeWOoouaI=', 'x-amz-request-id': '4T00ZMXSBKYAPNYK', 'date': 'Fri, 13 Feb 2026 21:20:39 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'audio-catalog-bucket-2026', 'CreationDate': datetime.datetime(2026, 2, 13, 5, 7, 8, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::audio-catalog-bucket-2026'}, {'Name': 'justin-music-ai-experiments-2026', 'CreationDate': datetime.datetime(2026, 1, 28, 16, 48, 4, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::justin-music-ai-experiments-2026'}], 'Owner': {'ID': 'e0f927342bb02d014fad360924f168057ea105720f611024d85a517d3f34fb60'}}


In [8]:

paginator = client.get_paginator('list_objects_v2')
page_iterator = paginator.paginate(Bucket='justin-music-ai-experiments-2026')
filtered_iterator = page_iterator.search("Contents[?Size > `100`][]")
for key_data in filtered_iterator: #iterates through dictionary of dictionaries
    #print(key_data) #prints all dictionary data
    print(key_data['Key']) #prints the value associated with the key

The Empty Heads - The Void and Beyond/Drained.aif
The Empty Heads - The Void and Beyond/Glass Half Full.aif
The Empty Heads - The Void and Beyond/Hardly There.aif
The Empty Heads - The Void and Beyond/Hole in the Bucket.aif
The Empty Heads - The Void and Beyond/Hot Air.aif
The Empty Heads - The Void and Beyond/I Have No Thoughts.aif
The Empty Heads - The Void and Beyond/Neutral Zone.aif
The Empty Heads - The Void and Beyond/So Empty.aif
The Empty Heads - The Void and Beyond/Spacing Out.aif
The Empty Heads - The Void and Beyond/Void.aif
